In [1]:
import pandas as pd

df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\game_tracking_2025.csv")

print("Shape:", df.shape)
display(df.head())
df.info(memory_usage="deep")

Shape: (11160903, 12)


,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2025090701,290,57347,2025-09-07T17:09:39.500,83.06,32.06,0.10,0.06,0.04,8.00,18.86,NaN
1,2025090701,290,57347,2025-09-07T17:09:39.600,83.07,32.07,0.10,0.07,0.02,2.61,21.95,NaN
2,2025090701,290,57347,2025-09-07T17:09:39.700,83.06,32.06,0.07,0.05,0.02,357.42,16.24,NaN
3,2025090701,290,57347,2025-09-07T17:09:39.800,83.07,32.07,0.07,0.05,0.01,352.73,18.93,NaN
4,2025090701,290,57347,2025-09-07T17:09:39.900,83.08,32.07,0.06,0.05,0.01,349.52,20.80,NaN


<class 'pandas.DataFrame'>
RangeIndex: 11160903 entries, 0 to 11160902
Data columns (total 12 columns):
 #   Column   Dtype  
---  ------   -----  
 0   game_id  int64  
 1   play_id  int64  
 2   nfl_id   int64  
 3   time     str    
 4   x        float64
 5   y        float64
 6   s        float64
 7   a        float64
 8   dis      float64
 9   o        float64
 10  dir      float64
 11  event    str    
dtypes: float64(7), int64(3), str(2)
memory usage: 1.9 GB


In [2]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(missing_summary)

,missing_count,missing_pct
game_id,0,0.00
play_id,0,0.00
nfl_id,0,0.00
time,0,0.00
x,0,0.00
y,0,0.00
s,0,0.00
a,0,0.00
dis,0,0.00
o,0,0.00


### Missing Values Analysis — game_tracking_2025

**Findings:**
- All identifier, timestamp, position, and movement columns have 0% missing values.
- The `event` column contains 10,425,238 missing values (93.41%).
- Event missingness is consistent across the three seasons:
  - 2023: 93.36%
  - 2024: 93.34%
  - 2025: 93.41%

**Interpretation:**
- Missing event annotations are likely structural because events are recorded at specific timestamps rather than every tracking frame.
- No missing values were detected in the remaining 11 columns.

**Preprocessing Decision:**
- Preserve all tracking observations.
- Retain missing `event` values.
- Avoid automatic row deletion or event imputation.

**Conclusion:**
The 2025 tracking dataset is complete for identifiers and movement measurements. High event missingness is consistent with the previous seasons and is not necessarily a data quality issue.

In [3]:
candidate_key = ["game_id", "play_id", "nfl_id", "time"]

print("Full Duplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Candidate Keys:",
    df.duplicated(subset=candidate_key).sum()
)

print(
    "Missing Values in Candidate Key:",
    df[candidate_key].isna().sum().sum()
)

Full Duplicate Rows: 0
Duplicate Candidate Keys: 0
Missing Values in Candidate Key: 0


### Granularity and Unique Identifier Validation — game_tracking_2025

**Granularity:** One tracking observation per player per timestamp within a game play.

**Validated Candidate Composite Key:** `(game_id, play_id, nfl_id, time)`

**Validation Results:**
- Full duplicate rows: 0
- Duplicate candidate keys: 0
- Missing values in candidate key columns: 0

**Conclusion:**
The candidate composite key uniquely identifies every tracking observation in the 2025 dataset. No duplicate records or null candidate keys were detected. Results are consistent across all three tracking seasons (2023–2025).

In [4]:
print("Unique Games:", df["game_id"].nunique())

print(
    "Unique Plays:",
    df[["game_id", "play_id"]].drop_duplicates().shape[0]
)

print("Unique Players:", df["nfl_id"].nunique())

display(df["event"].value_counts(dropna=False).head(20))

Unique Games: 334
Unique Plays: 42544
Unique Players: 448


event
NaN                          10425238
ball_snap                      154839
first_contact                   91278
pass_forward                    91231
tackle                          91186
pass_arrived                    68478
pass_outcome_caught             54929
handoff                         46564
pass_outcome_incomplete         31977
play_action                     25740
out_of_bounds                   16484
autoevent_passinterrupted       11209
run                              7513
qb_sack                          6039
man_in_motion                    5653
line_set                         4207
touchdown                        4016
play_submit                      3869
dropped_pass                     2971
pass_outcome_touchdown           2178
Name: count, dtype: int64

In [5]:
numeric_cols = ["x", "y", "s", "a", "dis", "o", "dir"]

display(df[numeric_cols].describe().T)

print("Negative Speed:", (df["s"] < 0).sum())
print("Negative Distance:", (df["dis"] < 0).sum())

,count,mean,std,min,25%,50%,75%,max
x,11160903.0,59.987644,24.554064,-5.860,41.81,59.98,78.25,127.400
y,11160903.0,26.737899,9.133221,-8.780,21.33,26.75,32.13,63.050
s,11160903.0,2.471210,2.049428,0.000,0.81,2.05,3.68,16.880
a,11160903.0,1.553226,1.199302,-14.396,0.64,1.34,2.22,33.759
dis,11160903.0,0.251626,0.206102,0.000,0.08,0.21,0.37,4.200
o,11160903.0,178.968779,100.476843,0.000,89.52,179.17,269.30,360.000
dir,11160903.0,180.085108,103.193111,0.000,90.46,179.87,269.69,360.000


Negative Speed: 0
Negative Distance: 0


In [6]:
print("Negative Acceleration Records:", (df["a"] < 0).sum())

print(
    "Negative Acceleration Percentage:",
    round((df["a"] < 0).mean() * 100, 4)
)

Negative Acceleration Records: 82
Negative Acceleration Percentage: 0.0007


### Negative Acceleration Investigation — game_tracking_2025

**Column:** `a` — Instantaneous player acceleration (yds/sec²).

**Findings:**
- Negative acceleration records: 82
- Percentage of total records: 0.0007%
- Minimum acceleration: -14.396
- No negative acceleration values were observed in the 2023 or 2024 tracking datasets.

**Interpretation:**
Negative acceleration may be valid for signed acceleration measurements. However, the column description does not explicitly establish whether `a` represents signed acceleration or acceleration magnitude.

**Preprocessing Decision:**
- Retain all 82 records.
- Flag negative acceleration values as potential anomalies.
- Verify the measurement convention before modifying or removing these observations.

**Conclusion:**
Negative acceleration affects a very small proportion of the 2025 tracking dataset. Its validity remains unconfirmed and requires further investigation.

In [7]:
import pandas as pd

games_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\games.csv")
players_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\players.csv")

# 1. Check tracking game IDs against games.csv
print(
    "Tracking Game IDs missing from games:",
    len(set(df["game_id"]) - set(games_df["game_id"]))
)

# 2. Check tracking player IDs against players.csv
print(
    "Tracking Player IDs missing from players:",
    len(set(df["nfl_id"]) - set(players_df["nfl_id"]))
)

# 3. Check 2025 games without tracking
missing_games_2025 = (
    set(games_df.loc[games_df["season"] == 2025, "game_id"])
    - set(df["game_id"])
)

print("2025 Games without tracking:", len(missing_games_2025))
print("Missing Game IDs:", missing_games_2025)

Tracking Game IDs missing from games: 0
Tracking Player IDs missing from players: 0
2025 Games without tracking: 0
Missing Game IDs: set()


# Data Profiling Summary — game_tracking_2025.csv

### 1. Dataset Overview
- Total records: 11,160,903
- Total columns: 12
- Unique games: 334
- Unique game-play combinations: 42,544
- Unique players: 448

### 2. Missing Values
- All identifier, timestamp, position, and movement columns have 0% missing values.
- The `event` column contains 10,425,238 missing values (93.41%).
- Missing event annotations are likely structural and consistent with previous seasons.

### 3. Granularity and Uniqueness
- Granularity: One tracking observation per player per timestamp within a game play.
- Validated candidate composite key: `(game_id, play_id, nfl_id, time)`
- Full duplicate rows: 0
- Duplicate composite keys: 0
- Missing composite key values: 0

### 4. Categorical and Numeric Validation
- No negative speed or distance values were detected.
- Orientation and direction range from 0 to 360 degrees.
- Maximum speed: 16.88
- Maximum acceleration: 33.759
- Maximum distance: 4.20
- Negative acceleration records: 82 (0.0007%).

**Potential Issue:** Negative acceleration values require further investigation to confirm whether the field represents signed acceleration or acceleration magnitude.

### 5. Referential Integrity
- Tracking game IDs missing from `games.csv`: 0
- Tracking player IDs missing from `players.csv`: 0
- 2025 games without tracking: 0

All 334 games from the 2025 season are represented in the tracking dataset.

### 6. Recommended Preprocessing Actions
- Preserve missing `event` values.
- Retain all tracking frames.
- Flag negative acceleration and extreme movement values for review.
- Use the validated composite key for frame-level uniqueness checks.
- Validate player-play linkage separately before joining with `player_play.csv`.

### Conclusion
The 2025 tracking dataset passes the core completeness, uniqueness, and game/player referential integrity checks. All 2025 games are covered. Negative acceleration values and potential extreme movement measurements remain documented for further investigation.